In [36]:
import numpy as np
import pandas as pd

# Define parameters for slide grid calculation
# Strikes (x-axis)
strikes = np.array([70, 80.0, 82.5, 85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100.0, 102.5, 105.0, 107.5, 110.0, 112.5, 115.0, 117.5, 120.0])

# Tenors/Maturities (y-axis) in years
tenors = np.array([1/252,0.08, 0.25, 0.50, 1.00, 2.00])

# Option parameters
F = 100.0          # Forward price
K = 100.0          # Strike price (reference)
r = 0.05           # Risk-free rate
sigma = 0.20       # Volatility
option_type = 'put'

print("Parameters defined:")
print(f"Strikes: {strikes}")
print(f"Tenors: {tenors}")
print(f"Forward (F): {F}, Rate (r): {r}, Volatility (σ): {sigma}")
print(f"Option Type: {option_type}")

Parameters defined:
Strikes: [ 70.   80.   82.5  85.   87.5  90.   92.5  95.   97.5 100.  102.5 105.
 107.5 110.  112.5 115.  117.5 120. ]
Tenors: [0.00396825 0.08       0.25       0.5        1.         2.        ]
Forward (F): 100.0, Rate (r): 0.05, Volatility (σ): 0.2
Option Type: put


In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('/Users/kevinlaventure/python_code')
from python_module.pricing_model import BSMModel

# Parameters (using existing kernel variables)
# F, r, sigma, option_type should be defined in kernel

# Create slide grid with -30% slide and 1/252 time bump
slide_value = -0.30
time_bump = 1/252

slide_grid = pd.DataFrame(
    index=tenors,
    columns=strikes,
    dtype=float
)

for tenor in tenors:
    for strike in strikes:
        # Base case
        base = BSMModel.compute_option(
            F=F, K=strike, T=tenor, r=r, sigma=sigma, 
            option_type=option_type, compute_greeks=True
        )
        
        # Slide case with time bump
        bumped_tenor = tenor + time_bump
        slide_result = BSMModel.compute_option(
            F=F, K=strike, T=bumped_tenor, r=r, sigma=sigma,
            option_type=option_type, compute_greeks=True,
            slide_scenario=[slide_value],
            slide_compute='option_pnl'
        )
        
        # Extract the slide PnL
        slide_grid.loc[tenor, strike] = slide_result[slide_value]

# Convert to numeric for cleaner display
slide_grid = slide_grid.astype(float)

print("Slide Grid (-30% slide with 1/252 time bump):")
print(slide_grid.round(4))

Slide Grid (-30% slide with 1/252 time bump):
           70.0     80.0     82.5     85.0     87.5     90.0     92.5   \
0.003968  0.4974   9.9960  12.4950  14.9940  17.4931  19.9921  22.4911   
0.080000  1.6114   9.9737  12.4500  14.9336  17.4072  19.8415  22.1764   
0.250000  2.7776  10.1533  12.4082  14.6830  16.9142  19.0434  21.0186   
0.500000  3.8407  10.4082  12.3239  14.2466  16.1282  17.9271  19.6098   
1.000000  5.0749  10.5258  12.0130  13.4953  14.9483  16.3512  17.6869   
2.000000  6.0740  10.1930  11.2577  12.3133  13.3502  14.3601  15.3360   

            95.0     97.5     100.0    102.5    105.0    107.5    110.0  \
0.003968  24.9891  27.4275  29.2776  29.9199  29.9864  29.9881  29.9881   
0.080000  24.3148  26.1428  27.5723  28.5799  29.2146  29.5704  29.7478   
0.250000  22.7965  24.3465  25.6533  26.7178  27.5552  28.1915  28.6588   
0.500000  21.1516  22.5365  23.7571  24.8136  25.7120  26.4633  27.0815   
1.000000  18.9423  20.1082  21.1792  22.1529  23.0297  23.81

In [38]:
# Visualize the slide grid using Plotly
import plotly.graph_objects as go

# Create interactive heatmap
fig = go.Figure(
    data=go.Heatmap(
        z=slide_grid.values,
        x=[f'{x:.1f}' for x in slide_grid.columns],
        y=[f'{y:.4f}' for y in slide_grid.index],
        colorscale='RdYlGn',
        text=slide_grid.round(4).values,
        texttemplate='%{text:.4f}',
        textfont={"size": 10},
        colorbar=dict(title="PnL", thickness=20, len=0.7),
        hovertemplate='Strike: %{x}<br>Tenor: %{y}<br>PnL: %{text:.4f}<extra></extra>'
    )
)

fig.update_layout(
    title=f'Slide Grid: -30% Forward Move with 1/252 Time Bump<br><sub>Option Type: {option_type}</sub>',
    xaxis_title='Strike',
    yaxis_title='Tenor (Maturity)',
    width=1200,
    height=600,
    font=dict(size=11),
    hovermode='closest'
)

fig.show()

# Print the grid as a table for detailed view
print("\nSlide Grid Values (rounded to 4 decimals):")
print(slide_grid.round(4).to_string())

print(f"\n{'='*60}")
print(f"Grid Summary Statistics:")
print(f"{'='*60}")
print(f"Min:     {slide_grid.values.min():.6f}")
print(f"Max:     {slide_grid.values.max():.6f}")
print(f"Mean:    {slide_grid.values.mean():.6f}")
print(f"Std Dev: {slide_grid.values.std():.6f}")
print(f"Shape:   {slide_grid.shape} (tenors × strikes)")


Slide Grid Values (rounded to 4 decimals):
           70.0     80.0     82.5     85.0     87.5     90.0     92.5     95.0     97.5     100.0    102.5    105.0    107.5    110.0    112.5    115.0    117.5    120.0
0.003968  0.4974   9.9960  12.4950  14.9940  17.4931  19.9921  22.4911  24.9891  27.4275  29.2776  29.9199  29.9864  29.9881  29.9881  29.9881  29.9881  29.9881  29.9881
0.080000  1.6114   9.9737  12.4500  14.9336  17.4072  19.8415  22.1764  24.3148  26.1428  27.5723  28.5799  29.2146  29.5704  29.7478  29.8267  29.8581  29.8693  29.8729
0.250000  2.7776  10.1533  12.4082  14.6830  16.9142  19.0434  21.0186  22.7965  24.3465  25.6533  26.7178  27.5552  28.1915  28.6588  28.9906  29.2188  29.3710  29.4694
0.500000  3.8407  10.4082  12.3239  14.2466  16.1282  17.9271  19.6098  21.1516  22.5365  23.7571  24.8136  25.7120  26.4633  27.0815  27.5823  27.9820  28.2966  28.5408
1.000000  5.0749  10.5258  12.0130  13.4953  14.9483  16.3512  17.6869  18.9423  20.1082  21.1792  22.1529

In [39]:
# Define two options for combination/spread analysis
# Option 1
option1_type = 'call'
option1_strike = 95.0
option1_quantity = 1  # +1 = long, -1 = short

# Option 2
option2_type = 'call'
option2_strike = 105.0
option2_quantity = -1  # +1 = long, -1 = short

# Grid parameters for spot bump analysis
spot_bumps = np.array([-0.3, -0.1, -0.05, 0, 0.05, 0.1, 0.3])  # -30% to +30% in 1% increments

print("Option Combination Parameters:")
print(f"Option 1: {option1_quantity:+.0f}x {option1_type.upper()} @ {option1_strike}")
print(f"Option 2: {option2_quantity:+.0f}x {option2_type.upper()} @ {option2_strike}")
print(f"\nSpot Bumps: {spot_bumps.min():.1%} to {spot_bumps.max():.1%}")
print(f"Number of bumps: {len(spot_bumps)}")

Option Combination Parameters:
Option 1: +1x CALL @ 95.0
Option 2: -1x CALL @ 105.0

Spot Bumps: -30.0% to 30.0%
Number of bumps: 7


In [40]:
# Calculate combination grid with spot bumps
# Use tenor from 1 year as reference
reference_tenor = 1.0
time_bump = 1/252  # One day

# Create single-row grid for the combination
combination_grid = pd.DataFrame(
    index=['Combination'],
    columns=spot_bumps,
    dtype=float
)

for bump in spot_bumps:
    # Bumped forward
    F_bumped = F * (1 + bump)
    bumped_tenor = reference_tenor + time_bump
    
    # Option 1 PnL
    option1_base = BSMModel.compute_option(
        F=F, K=option1_strike, T=reference_tenor, r=r, sigma=sigma,
        option_type=option1_type, compute_greeks=False
    )
    option1_bumped = BSMModel.compute_option(
        F=F_bumped, K=option1_strike, T=bumped_tenor, r=r, sigma=sigma,
        option_type=option1_type, compute_greeks=False
    )
    # Extract prices - handle both dict and scalar returns
    price1_base = option1_base['price'] if isinstance(option1_base, dict) else option1_base
    price1_bumped = option1_bumped['price'] if isinstance(option1_bumped, dict) else option1_bumped
    option1_pnl = (price1_bumped - price1_base) * option1_quantity
    
    # Option 2 PnL
    option2_base = BSMModel.compute_option(
        F=F, K=option2_strike, T=reference_tenor, r=r, sigma=sigma,
        option_type=option2_type, compute_greeks=False
    )
    option2_bumped = BSMModel.compute_option(
        F=F_bumped, K=option2_strike, T=bumped_tenor, r=r, sigma=sigma,
        option_type=option2_type, compute_greeks=False
    )
    # Extract prices - handle both dict and scalar returns
    price2_base = option2_base['price'] if isinstance(option2_base, dict) else option2_base
    price2_bumped = option2_bumped['price'] if isinstance(option2_bumped, dict) else option2_bumped
    option2_pnl = (price2_bumped - price2_base) * option2_quantity
    
    # Combined PnL
    combination_grid.loc['Combination', bump] = option1_pnl + option2_pnl

# Convert to numeric
combination_grid = combination_grid.astype(float)

print("Combination Grid (Spot Bump Analysis):")
print(combination_grid.round(4).to_string())
print(f"\nGrid shape: {combination_grid.shape}")
print(f"PnL Range: {combination_grid.values.min():.4f} to {combination_grid.values.max():.4f}")

Combination Grid (Spot Bump Analysis):
             -0.30   -0.10   -0.05    0.00    0.05    0.10    0.30
Combination -4.089 -1.8357 -0.9366 -0.0016  0.9101  1.7514  4.0247

Grid shape: (1, 7)
PnL Range: -4.0890 to 4.0247


In [42]:
# Visualize combination grid using Plotly
import plotly.graph_objects as go

# Create 1D heatmap (single row)
fig = go.Figure(
    data=go.Heatmap(
        z=combination_grid.values,
        x=[f'{x:.1%}' for x in spot_bumps],
        y=['Combination'],
        colorscale='RdYlGn',
        text=combination_grid.round(4).values,
        texttemplate='%{text:.4f}',
        textfont={"size": 11},
        colorbar=dict(title="PnL", thickness=20, len=0.7),
        hovertemplate='Spot Bump: %{x}<br>PnL: %{text:.4f}<extra></extra>',
        zmin=combination_grid.values.min(),
        zmax=combination_grid.values.max()
    )
)

fig.update_layout(
    title=f'Option Combination PnL vs Spot Bump (with 1/252 Time Bump)<br><sub>' + 
          f'{option1_quantity:+.0f}x {option1_type.upper()}@{option1_strike} + ' +
          f'{option2_quantity:+.0f}x {option2_type.upper()}@{option2_strike} | ' +
          f'Tenor: {reference_tenor:.2f}y → {reference_tenor + time_bump:.4f}y, σ: {sigma:.1%}</sub>',
    xaxis_title='Spot Bump',
    yaxis_title='',
    width=1400,
    height=300,
    font=dict(size=11),
    hovermode='x unified',
    yaxis=dict(automargin=True)
)

fig.show()

# Summary statistics
print(f"\n{'='*60}")
print(f"Combination Summary Statistics:")
print(f"{'='*60}")
print(f"Max PnL:    {combination_grid.values.max():.6f} at {spot_bumps[combination_grid.values.argmax()]:.1%}")
print(f"Min PnL:    {combination_grid.values.min():.6f} at {spot_bumps[combination_grid.values.argmin()]:.1%}")
print(f"PnL @ 0%:   {combination_grid.iloc[0, np.argmin(np.abs(spot_bumps))]:.6f}")
print(f"Mean PnL:   {combination_grid.values.mean():.6f}")
print(f"Std Dev:    {combination_grid.values.std():.6f}")


Combination Summary Statistics:
Max PnL:    4.024703 at 30.0%
Min PnL:    -4.089035 at -30.0%
PnL @ 0%:   -0.001642
Mean PnL:   -0.025254
Std Dev:    2.421839
